# Train ReDet with Blur Augmentation

This notebook demonstrates the complete training workflow for pedicle detection with blur augmentation using the `blur_robust` pipeline.

**Workflow:**
1. Environment check
2. Register custom datasets (`PEdataset`, `VBdataset`)
3. Import and register blur augmentation transforms
4. Load enhanced config with `get_blur_robust_pedicle_config()`
5. Show config diff (before/after blur augmentation)
6. Create Runner and start training
7. Validate and test

## 1. Environment Check

In [ ]:
import sys
import torch
import mmcv
import mmdet

print(f'Python   : {sys.version}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'mmcv     : {mmcv.__version__}')
print(f'mmdet    : {mmdet.__version__}')

try:
    import mmrotate
    print(f'mmrotate : {mmrotate.__version__}')
except ImportError:
    print('mmrotate : NOT installed — please install mmrotate 1.0.0rc1')

## 2. Register Custom Datasets

In [ ]:
from mmdet.registry import DATASETS
from mmrotate.datasets import DOTADataset

# ---- PEdataset (pedicle) ----
@DATASETS.register_module(name='PEdataset', force=True)
class PEdataset(DOTADataset):
    """Pedicle detection dataset (DOTA-format annotations)."""
    METAINFO = {
        'classes': ('pedicle',),
        'palette': [(220, 20, 60)],
    }

# ---- VBdataset (vertebra) ----
@DATASETS.register_module(name='VBdataset', force=True)
class VBdataset(DOTADataset):
    """Vertebra detection dataset (DOTA-format annotations)."""
    METAINFO = {
        'classes': ('VB',),
        'palette': [(0, 128, 0)],
    }

print('Datasets registered: PEdataset, VBdataset')

## 3. Import and Register Blur Augmentation Transforms

In [ ]:
import sys
import os

# Add repository root to path if needed
repo_root = os.path.dirname(os.path.abspath('.'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Importing blur_augmentation will auto-register the transforms via decorators
from blur_robust.blur_augmentation import BlurAugmentation, AdaptiveSharpen

from mmdet.registry import TRANSFORMS
print('BlurAugmentation registered:', 'BlurAugmentation' in TRANSFORMS._module_dict)
print('AdaptiveSharpen  registered:', 'AdaptiveSharpen'  in TRANSFORMS._module_dict)

## 4. Load Enhanced Config

In [ ]:
from blur_robust.enhanced_config import get_blur_robust_pedicle_config

cfg_dict = get_blur_robust_pedicle_config()
print('Config loaded successfully.')
print('dataset_type :', cfg_dict['dataset_type'])
print('work_dir     :', cfg_dict['work_dir'])
print('optimizer lr :', cfg_dict['optim_wrapper']['optimizer']['lr'])

## 5. Config Diff — Before vs. After Blur Augmentation

In [ ]:
import pprint

print('=== Train pipeline (blur-robust) ===')
for step in cfg_dict['train_pipeline']:
    print(' -', step.get('type'), step.get('prob', ''))

print()
print('=== Scheduler ===')
pprint.pprint(cfg_dict['param_scheduler'])

print()
print('=== NMS (test_cfg.rcnn) ===')
pprint.pprint(cfg_dict['model']['test_cfg']['rcnn']['nms'])

### Compare: Original pipeline vs. blur-robust pipeline

In [ ]:
original_pipeline = [
    {'type': 'mmdet.LoadImageFromFile'},
    {'type': 'mmdet.LoadAnnotations', 'box_type': 'qbox', 'with_bbox': True},
    {'type': 'ConvertBoxType', 'box_type_mapping': {'gt_bboxes': 'rbox'}},
    {'type': 'mmdet.Resize', 'keep_ratio': True, 'scale': (1024, 1024)},
    {'type': 'mmdet.RandomFlip', 'direction': ['horizontal', 'vertical', 'diagonal'], 'prob': 0.75},
    {'type': 'mmdet.PackDetInputs'},
]

blur_robust_pipeline = cfg_dict['train_pipeline']

print('Original pipeline steps:')
for i, step in enumerate(original_pipeline):
    print(f'  [{i}] {step["type"]}')

print()
print('Blur-robust pipeline steps:')
for i, step in enumerate(blur_robust_pipeline):
    marker = ' <-- NEW' if step['type'] in ('BlurAugmentation', 'AdaptiveSharpen') else ''
    print(f'  [{i}] {step["type"]}{marker}')

## 6. Create Runner and Start Training

In [ ]:
from mmengine.config import Config
from mmengine.runner import Runner

# Convert the config dict to an mmengine Config object
cfg = Config(cfg_dict)

# Create the runner
runner = Runner.from_cfg(cfg)

print('Runner created. Starting training...')
runner.train()

## 7. Validate and Test

In [ ]:
# Run validation on the best checkpoint
# Update 'load_from' to point to your trained checkpoint
cfg_val = Config(cfg_dict)
cfg_val.load_from = os.path.join(cfg_dict['work_dir'], 'epoch_10.pth')  # adjust as needed

val_runner = Runner.from_cfg(cfg_val)
print('Running validation...')
val_runner.val()

## (Optional) Quick Deblurring Demo

In [ ]:
import cv2
import matplotlib.pyplot as plt
from blur_robust.deblur_preprocess import DeblurPreprocessor, assess_blur_level

# Replace with an actual X-ray image path
SAMPLE_IMAGE = 'val_images/sample.jpg'

image = cv2.imread(SAMPLE_IMAGE)
if image is None:
    print(f'Image not found: {SAMPLE_IMAGE} — skipping demo')
else:
    preprocessor = DeblurPreprocessor()
    result = preprocessor.process(image)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Original\nBlur: {result["blur_level"]} (var={result["blur_var"]:.1f})')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(result['image'], cv2.COLOR_BGR2RGB))
    axes[1].set_title('After Deblurring Pipeline')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
    print(f'Wiener deconvolution applied: {result["wiener_applied"]}')